# Pose Detection with AlphaPhose v0.3.0

This notebook uses an open source project [MVIG-SJTU/AlphaPose](https://github.com/MVIG-SJTU/AlphaPose) to detect/track multi person poses on a given youtube video.

For other deep-learning Colab notebooks, visit [tugstugi/dl-colab-notebooks](https://github.com/tugstugi/dl-colab-notebooks).


## Install AlphaPose v0.3.0

In [ ]:
import os
import sys

# 0. PROJECT PATH
project_name = '/content/AlphaPose'

# 1. FIX PATHS (Essential for finding the CUDA compiler 'nvcc')
os.environ['PATH'] = "/usr/local/cuda/bin/:" + os.environ.get('PATH', '')
os.environ['LD_LIBRARY_PATH'] = "/usr/local/cuda/lib64/:" + os.environ.get('LD_LIBRARY_PATH', '')

# 2. GET ALPHAPOSE
%cd /content
if not os.path.exists(project_name):
    !git clone https://github.com/MVIG-SJTU/AlphaPose.git

# 3. INSTALL DEPENDENCIES
!pip install -q cython cython_bbox
!sudo apt-get install -y -q libyaml-dev

# 4. BUILD & INSTALL
%cd {project_name}
!python3 setup.py build develop

# 5. INSTALL PYTORCH3D
!pip install -q fvcore iopath
!pip install "git+https://github.com/facebookresearch/pytorch3d.git"

# 6. VERIFY THE INSTALL
sys.path.append(project_name)
try:
    import alphapose
    print("\n✅ AlphaPose Pip Setup Successful!")
except Exception as e:
    print(f"\n❌ Setup failed: {e}")

%cd /content

/content
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/timm-0.1.20-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
DEPRECATION: Loading egg at /usr/local/lib/python3.12/dist-packages/munkres-1.1.4-py3.12.egg is deprecated. pip 24.3 will enforce this behaviour change. A possible replacement is to use pip for package installation. Discussion can be found at https://github.com/pypa/pip/issues/12330
Reading package lists...
Building dependency tree...
Reading state information...
libyaml-dev is already the newest version (0.2.2-1build2).
0 upgraded, 0 newly installed, 0 to remove and 45 not upgraded.
/content/AlphaPose
/usr/local/lib/python3.12/dist-packages/setuptools/__init__.py:94: _DeprecatedInstaller: setuptools.installer and fetch_build_eggs are deprecated.
!!

        ******************************************

## Download pretrained models

In [24]:
# 1. Setup YOLOv3-SPP Detector Weights
yolo_pretrained_model_path = join(project_name, 'detector/yolo/data/yolov3-spp.weights')
if not exists(yolo_pretrained_model_path):
  !mkdir -p {project_name}/detector/yolo/data
  !gdown -O {yolo_pretrained_model_path} https://drive.google.com/uc?id=1D47msNOOiJKvPOXlnpyzdKA3k6E97NTC

# 2. Setup FastPose ResNet50 (Halpe 26 Keypoints)
pretrained_model_path = join(project_name, 'pretrained_models/halpe26_fast_res50_256x192.pth')
pretrained_model_config_path = join(project_name, 'configs/halpe_26/resnet/256x192_res50_lr1e-3_1x.yaml')
if not exists(pretrained_model_path):
  !mkdir -p {project_name}/pretrained_models
  # Downloading the Halpe 26 model weights
  !gdown -O {pretrained_model_path} https://drive.google.com/uc?id=1S-ROA28de-1zvLv-hVfPFJ5tFBYOSITb

print("✅ Models and configurations are ready.")

Downloading...
From (original): https://drive.google.com/uc?id=1S-ROA28de-1zvLv-hVfPFJ5tFBYOSITb
From (redirected): https://drive.google.com/uc?id=1S-ROA28de-1zvLv-hVfPFJ5tFBYOSITb&confirm=t&uuid=4dd559c2-8a87-45c2-be54-82bd4637eae4
To: /content/AlphaPose/AlphaPose/pretrained_models/halpe26_fast_res50_256x192.pth
100% 163M/163M [00:06<00:00, 27.0MB/s]
✅ Models and configurations are ready.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [29]:
import os
import sys

# 1. SET FILENAMES
# Make sure your video is actually in the 'assets' folder
INPUT_FILE = "/content/assets/squats.mp4"
PROJECT_DIR = "/content/AlphaPose"

# 2. FIX ENVIRONMENT
if os.path.exists(PROJECT_DIR):
    sys.path.append(PROJECT_DIR)
    os.environ['PYTHONPATH'] = PROJECT_DIR
else:
    print(f"❌ Error: {PROJECT_DIR} folder not found. Run your setup cell first!")

# 3. PREPARE VIDEO (NO TRIMMING)
# We still run it through ffmpeg once to ensure the codec is standard (libx264)
# but we removed the '-t' and '-ss' flags to keep the full duration.
if os.path.exists(INPUT_FILE):
    !ffmpeg -y -loglevel error -i {INPUT_FILE} -c:v libx264 /content/full_input.mp4
    print("🎥 Video prepared for processing (Full Duration).")
else:
    raise FileNotFoundError(f"Could not find {INPUT_FILE}")

# 4. RUN ALPHAPOSE
print("🚀 Starting AlphaPose Inference on full video...")
# Note: Full videos take longer; ensure you are on a GPU runtime!
!python3 /content/AlphaPose/scripts/demo_inference.py --sp \
  --cfg /content/AlphaPose/configs/halpe_26/resnet/256x192_res50_lr1e-3_1x.yaml \
  --checkpoint /content/AlphaPose/pretrained_models/halpe26_fast_res50_256x192.pth \
  --video /content/full_input.mp4 \
  --outdir /content/output/ \
  --save_video \
  --vis_fast

# 5. CONVERT AVI TO MP4
expected_avi = "/content/output/AlphaPose_full_input.avi"

if os.path.exists(expected_avi):
    !ffmpeg -y -loglevel error -i {expected_avi} -vcodec libx264 /content/AlphaPose_final.mp4
    print("✅ DONE! Your full video is ready at: /content/AlphaPose_final.mp4")
else:
    print("❌ AlphaPose failed to produce the output. Check for CUDA Out-of-Memory errors if the video is very long.")

🎥 Video prepared for processing (Full Duration).
🚀 Starting AlphaPose Inference on full video...
2026-03-28 01:55:49 [DEBUG]: Loaded backend agg version v2.2.
Loading YOLO model..
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Exception in thread Thread-2 (image_detection):
Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/lib/python3.

Finally, visualize the result:

In [ ]:
def show_local_mp4_video(file_name, width=640, height=480):
  import io
  import base64
  from IPython.display import HTML
  video_encoded = base64.b64encode(io.open(file_name, 'rb').read())
  return HTML(data='''<video width="{0}" height="{1}" alt="test" controls>
                        <source src="data:video/mp4;base64,{2}" type="video/mp4" />
                      </video>'''.format(width, height, video_encoded.decode('ascii')))

show_local_mp4_video('AlphaPose_video.mp4', width=960, height=720)